# Runtime benchmark: step throughput vs. environment size

Measures how long it takes to run a fixed number of `BlockchainEnv` steps as the
graph size (`num_nodes`) grows.

We separate three costs:
- **reset** — host-side network generation + partition (the `O(N^2)` distance matrix and `O(N^3)` MDS partition live here);
- **compile** — one-time XLA compilation of the jitted scan rollout;
- **run** — steady-state execution of `N_STEPS` jitted steps (the number that actually matters for training throughput).

Tune the knobs in the config cell, then *Run All*.

In [ ]:
import pathlib
import sys

# Make the src/ layout importable whether the notebook runs from analysis/ or the repo root.
_root = pathlib.Path.cwd()
while not (_root / "src" / "rl_blockchain").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root / "src"))

import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from rl_blockchain.env import BlockchainEnv, BlockchainEnvParams, RewardParams, stack_rewards
from rl_blockchain.graph import NetworkConfig

print("JAX backend:", jax.default_backend(), "| devices:", jax.devices())

In [ ]:
# --- Config ---------------------------------------------------------------
SIZES = [100, 250, 500, 1000, 2000]  # env sizes (num_nodes) to sweep
N_STEPS = 200                        # steps per rollout (the "certain number of steps")
REPEATS = 3                          # timed repeats (after compile) averaged
MAX_HANDLING_SIZE = 100              # cluster cap
VOTE_PROB = 0.2                      # dummy policy: probability a node votes
ROOT_SEED = 0                        # single root key for the whole benchmark

In [ ]:
def make_rollout(env, params, n_steps, vote_prob):
    """A jitted scan rollout of `n_steps` with a dummy random-voting policy."""

    @jax.jit
    def rollout(state, key):
        def body(carry, _):
            state, key = carry
            key, k_act, k_step = jax.random.split(key, 3)
            action = jax.random.bernoulli(k_act, vote_prob, (params.num_nodes,))
            ts = env.step(k_step, state, action, params)
            return (ts.state, key), ts.reward

        (final_state, _), rewards = jax.lax.scan(
            body, (state, key), None, length=n_steps
        )
        return final_state, rewards

    return rollout


def benchmark_size(num_nodes, key):
    """Time reset, compile, and steady-state run for one env size."""
    env = BlockchainEnv()
    params = BlockchainEnvParams(
        network=NetworkConfig(num_nodes=num_nodes),
        reward=RewardParams(horizon=20),
        max_handling_size=MAX_HANDLING_SIZE,
        max_steps=N_STEPS,
    )

    # reset (host-side: network gen + partition)
    key, k_reset = jax.random.split(key)
    t0 = time.perf_counter()
    _, state = env.reset(k_reset, params)
    jax.block_until_ready(state.graph.trust_rating)
    reset_s = time.perf_counter() - t0

    rollout = make_rollout(env, params, N_STEPS, VOTE_PROB)
    n_clusters = int(state.cluster_batch.max_clusters)

    # compile (first call) — timed separately
    key, k_run = jax.random.split(key)
    t0 = time.perf_counter()
    _, rewards = rollout(state, k_run)
    jax.block_until_ready(rewards)
    compile_s = time.perf_counter() - t0

    # steady-state run (compiled), averaged over REPEATS
    t0 = time.perf_counter()
    for _ in range(REPEATS):
        _, rewards = rollout(state, k_run)
    jax.block_until_ready(rewards)
    run_s = (time.perf_counter() - t0) / REPEATS

    return {
        "num_nodes": num_nodes,
        "n_clusters": n_clusters,
        "reset_s": reset_s,
        "compile_s": compile_s,
        "run_s": run_s,
        "per_step_ms": run_s / N_STEPS * 1e3,
        "steps_per_s": N_STEPS / run_s,
    }

In [ ]:
key = jax.random.PRNGKey(ROOT_SEED)
results = []
for n in SIZES:
    key, k = jax.random.split(key)
    res = benchmark_size(n, k)
    results.append(res)
    print(
        f"N={res['num_nodes']:>5}  K={res['n_clusters']:>3}  "
        f"reset={res['reset_s']*1e3:8.1f} ms  compile={res['compile_s']*1e3:8.1f} ms  "
        f"run({N_STEPS})={res['run_s']*1e3:8.1f} ms  "
        f"per_step={res['per_step_ms']:6.3f} ms  ({res['steps_per_s']:8.0f} steps/s)"
    )

In [ ]:
sizes = [r["num_nodes"] for r in results]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: total time for N_STEPS, broken down.
axes[0].plot(sizes, [r["run_s"] * 1e3 for r in results], "o-", label=f"run ({N_STEPS} steps)")
axes[0].plot(sizes, [r["compile_s"] * 1e3 for r in results], "s--", label="compile (one-time)")
axes[0].plot(sizes, [r["reset_s"] * 1e3 for r in results], "^--", label="reset (one-time)")
axes[0].set_xlabel("env size (num_nodes)")
axes[0].set_ylabel("time (ms)")
axes[0].set_title(f"Runtime vs env size  ({N_STEPS} steps)")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].grid(True, which="both", alpha=0.3)
axes[0].legend()

# Right: steady-state per-step latency.
axes[1].plot(sizes, [r["per_step_ms"] for r in results], "o-", color="tab:green")
axes[1].set_xlabel("env size (num_nodes)")
axes[1].set_ylabel("per-step latency (ms)")
axes[1].set_title("Steady-state per-step cost")
axes[1].set_xscale("log")
axes[1].grid(True, which="both", alpha=0.3)

fig.tight_layout()
plt.show()

## Notes

- **`run`** is the throughput that matters for training; it should scale roughly linearly with `num_nodes` (the per-step cost is dominated by the `O(N)` feature/history updates and the `O(N^2)`-touching distance reward, which is a fused kernel).
- **`reset`** grows fastest: the default distance-based partitioner does an `O(N^3)` eigendecomposition. For very large graphs, switch to `BlockchainEnv(RouterPartitioner())` (O(N)).
- **`compile`** is paid once per (size, shape) and is independent of `N_STEPS`.
- On GPU, increase `SIZES` and `N_STEPS`; the per-step latency should stay flat until the device fills.